# Exercise 31.1: Exploring the Rice model

In this exercise, we will use the comprehensive muscle contraction model developed by **Rice et al. (2008)**. Because the internal mathematics of the coupled calcium-binding and crossbridge distortion states are highly complex, the model equations have been provided for you in an external file (`rice_model_2008.py`).

Our goal is to use this model to replicate classical cell mechanics experiments computationally.


## Exercise 31.1a: Steady-state Force-Calcium relations

The Rice model exhibits strong cooperativity. We will plot the steady-state Force-Calcium relation by holding the sarcomere length (SL) constant and simulating the model across a sequence of constant calcium concentrations until it reaches equilibrium.

Run the code below to generate the Force-Calcium curve.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import rice_model_2008 as rice  # External model file

# Generate an array of Calcium concentrations (µM)
Cai_array = np.logspace(-2, 1, 50)
Fss = np.zeros_like(Cai_array)

# Define time span for each simulation to reach steady state
t_span = (0, 500)
init_state = rice.init_state_values(SL=2.2)  # Fixed sarcomere length
force_index = rice.monitor_indices("active")

# Run the model for each calcium concentration
for i, Ca_level in enumerate(Cai_array):
    # Set the parameters (Constant Calcium)
    p = rice.init_parameter_values(
        start_time=2000, Ca_diastolic=Ca_level, SLmin=2.5, nperm=1
    )

    # We use the external RHS function
    sol = solve_ivp(rice.rhs, t_span, init_state, args=(p,), method="BDF")

    # Extract the steady-state force from the final time step
    final_state = sol.y[:, -1]
    m = rice.monitor(final_state, sol.t[-1], p)
    Fss[i] = m[force_index]

plt.figure(figsize=(7, 5))
plt.semilogx(Cai_array, Fss, color="dodgerblue", linewidth=2)
plt.ylabel("Normalized Force at Steady State")
plt.xlabel("Calcium Concentration (µM)")
plt.title("Steady-State Force-Calcium Relation")
plt.grid(alpha=0.3)
plt.show()

**Question:** Look at the plot generated above. What mathematical function does this curve resemble, and what does the steepness of the curve tell us about the biological behavior of the myofilaments?


## Exercise 31.1b: The isometric twitch

A classical cell mechanics experiment is the simple isometric twitch. A cell is held at a fixed length, stimulated with a physiological calcium transient, and the output force is measured over time.

Because the Rice model correctly couples the crossbridges to calcium, we can observe how stretching the cell (increasing the sarcomere length) drastically increases the peak force produced by the exact same calcium transient!

Fill in the code below to plot the isometric twitch for three different sarcomere lengths: 1.9 µM, 2.1 µM, and 2.3 µM.


In [ ]:
t_span = (0, 800)
t_eval = np.linspace(0, 800, 500)

# We use the default physiological calcium transient built into the parameters
p = rice.init_parameter_values(SLmin=2.5)
SL_values = [1.9, 2.1, 2.3]

plt.figure(figsize=(8, 5))

for SL in SL_values:
    # Initialize the state at the given sarcomere length
    init_state = rice.init_state_values(SL=SL)

    # Solve the ODE
    # sol = solve_ivp(...)

    # Extract the active force over time
    force = np.zeros_like(sol.t)
    for i in range(len(sol.t)):
        m = rice.monitor(sol.y[:, i], sol.t[i], p)
        force[i] = m[force_index]

    # Plot the result
    # plt.plot(...)

plt.xlabel("Time (ms)")
plt.ylabel("Active Force")
plt.title("Isometric Twitch at Different Sarcomere Lengths")
plt.legend()
plt.grid(alpha=0.3)
plt.show()